# Quality Control: Checking Our Selected Traces

Welcome back! We now have a folder (`selected_quick_migrate_mseed`) filled with perfectly curated waveform files for QuakeMigrate.

But before we hand this data over to a machine learning model, we must do some **Quality Control (QC)**. We need to answer two important questions:
1. **Are the files the right size?** Are they all roughly the same length in time and samples?
2. **Is there enough pre-event noise?** For an algorithm to detect a signal, it needs to establish a "baseline" of what normal background noise looks like. We need to ensure that our MiniSEED files start recording several seconds *before* the actual earthquake wave arrives.

Let's write a script to calculate these statistics for all 1,110 files!

### Step 1: Setting up and Loading the "Answers"
We need to know exactly when the waves arrived, so we load our CSV spreadsheet first. We store the arrival times in a dictionary so we can look them up instantly later.

In [1]:
import os
import glob
import csv
import numpy as np
from obspy import read, UTCDateTime

mseed_dir = 'selected_quick_migrate_mseed'
csv_file = 'filtered_picks_organized.csv'

print(f"Loading pick times from {csv_file}...")

# We store the picks in a dictionary grouped by station for fast lookup
picks_by_station = {}
with open(csv_file, 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        sta = row['station']
        try:
            pt = UTCDateTime(row['pick_time'])
            # If this is the first time seeing this station, create an empty list
            if sta not in picks_by_station:
                picks_by_station[sta] = []
            picks_by_station[sta].append(pt)
        except Exception:
            continue
            
print(f"Loaded picks for {len(picks_by_station)} stations.")

Loading pick times from filtered_picks_organized.csv...
Loaded picks for 13 stations.


### Step 2: Processing the Waveforms
Now we open every single MiniSEED file. For each file, we calculate its total length. 

Then, we find the exact wave arrival time (`pick_time`) that falls inside this file's recording window. We calculate the difference (`pick_time - starttime`) to see exactly how many seconds into the recording the wave hit.

In [2]:
mseed_files = glob.glob(os.path.join(mseed_dir, '*.mseed'))
print(f"Found {len(mseed_files)} MiniSEED files to check.\n")

# We will store all our calculations in these lists so we can do statistics later
all_lengths = []
all_samples = []
all_pick_diffs = []

for filepath in mseed_files:
    try:
        # Read the trace
        st = read(filepath)
        tr = st[0]
        
        sta = tr.stats.station
        start = tr.stats.starttime
        end = tr.stats.endtime
        
        # Calculate trace length (endtime minus starttime)
        length_sec = end - start
        samples = tr.stats.npts
        
        # Save these values to our lists
        all_lengths.append(length_sec)
        all_samples.append(samples)
        
        # Look up the pick time for this specific file
        valid_picks = [pt for pt in picks_by_station.get(sta, []) if start <= pt <= end]
        
        if valid_picks:
            # Calculate the difference (how much 'pre-event' time there is)
            diff_secs = [pt - start for pt in valid_picks]
            all_pick_diffs.extend(diff_secs)
            
    except Exception as e:
        pass

print("Finished processing all files!")

Found 1110 MiniSEED files to check.

Finished processing all files!


### Step 3: Comprehensive Statistical Summary
We use `numpy` (imported as `np`) to quickly calculate the Mean (average), Minimum, Maximum, and Standard Deviation for all of our files.

*Note: A low Standard Deviation means the files are very consistent!*

In [3]:
print(f"\n--- COMPREHENSIVE OVERALL SUMMARY ---")

print("\n[ Trace Lengths (seconds) ]")
print(f"  Mean: {np.mean(all_lengths):.2f}")
print(f"  Min:  {np.min(all_lengths):.2f}")
print(f"  Max:  {np.max(all_lengths):.2f}")
print(f"  Std:  {np.std(all_lengths):.2f}")

print("\n[ Trace Samples ]")
print(f"  Mean: {np.mean(all_samples):.0f}")
print(f"  Min:  {np.min(all_samples):.0f}")
print(f"  Max:  {np.max(all_samples):.0f}")
print(f"  Std:  {np.std(all_samples):.0f}")

if all_pick_diffs:
    print("\n[ Pick Differences (seconds from trace start) ]")
    print(f"  Mean: {np.mean(all_pick_diffs):.2f}")
    print(f"  Min:  {np.min(all_pick_diffs):.2f}")
    print(f"  Max:  {np.max(all_pick_diffs):.2f}")
    print(f"  Std:  {np.std(all_pick_diffs):.2f}")


--- COMPREHENSIVE OVERALL SUMMARY ---

[ Trace Lengths (seconds) ]
  Mean: 48.31
  Min:  41.80
  Max:  52.80
  Std:  5.41

[ Trace Samples ]
  Mean: 9663
  Min:  8361
  Max:  10561
  Std:  1081

[ Pick Differences (seconds from trace start) ]
  Mean: 7.67
  Min:  6.61
  Max:  10.13
  Std:  0.49


---
**Conclusion:**
By looking at the stats, we can confirm our dataset is incredibly healthy!
Our traces are all around 40-50 seconds long, and every single earthquake wave arrives roughly **7.6 seconds** after the recording starts (with a very low standard deviation of `0.49`). 

This means we have a consistent 7-second window of clean "background noise" to establish a baseline before the earthquake signals hit!